# MangaJaNai-Colab
### Manga AI upscaling · model tự động cho từng trang · resume trên Google Drive
Chọn **Runtime → Change runtime type → GPU**, và **Runtime Version: 2026.07 (Python 3.12)**, rồi chạy lần lượt sáu cell dưới đây.
Native backend `chainner_ext 0.3.10` chưa hỗ trợ Python 3.13; cell Install kiểm tra và báo hướng dẫn nếu chọn nhầm runtime.
Toàn bộ xử lý nằm trong GitHub; notebook chỉ tải source và gọi CLI.
[Hướng dẫn và source](https://github.com/dex593/MangaJaNai-Colab)

In [ ]:
# @title 1 · Cấu hình
GITHUB_REPO = "https://github.com/dex593/MangaJaNai-Colab.git" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}
INPUT_DIR = "/content/drive/MyDrive/Manga/Input" # @param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/Manga/Output" # @param {type:"string"}
MODEL_CACHE_DIR = "/content/drive/MyDrive/MangaJaNai/models" # @param {type:"string"}
UPSCALE_FACTOR = 2 # @param {type:"number"}
OUTPUT_FORMAT = "png" # @param ["png", "jpeg", "webp", "avif"]
AUTO_MODEL = True # @param {type:"boolean"}
RESUME = True # @param {type:"boolean"}
OVERWRITE = False # @param {type:"boolean"}
# @markdown ---
# @markdown **Nâng cao — giữ mặc định nếu chưa cần đổi.**
PIN_COMMIT = "" # @param {type:"string"}
GRAYSCALE_THRESHOLD = 12 # @param {type:"number"}
DETECTOR = "upstream" # @param ["upstream", "corrected"]
TILE_SIZE = 0 # @param {type:"integer"}
PRECISION = "auto" # @param ["auto", "fp32"]
STAGING = True # @param {type:"boolean"}
RETRY_FAILED = False # @param {type:"boolean"}
DRY_RUN = False # @param {type:"boolean"}
LOG_LEVEL = "normal" # @param ["quiet", "normal", "verbose", "debug"]
PROJECT_DIR = "/content/MangaJaNai-Colab"
print("Cấu hình đã sẵn sàng. 0 tile = tự ước lượng VRAM.")

In [ ]:
# @title 2 · Kết nối Google Drive
from google.colab import drive
from pathlib import Path
import os
import tempfile

drive.mount("/content/drive")
if not Path(INPUT_DIR).is_dir() or not os.access(INPUT_DIR, os.R_OK):
    raise ValueError(f"Không đọc được INPUT_DIR: {INPUT_DIR}. Tạo thư mục và chép ảnh vào Drive trước.")
for folder in (OUTPUT_DIR, MODEL_CACHE_DIR):
    Path(folder).mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=folder, prefix=".manga-probe-") as probe:
        probe.write(b"ok")
        probe.flush()
        os.fsync(probe.fileno())
print("Drive đã kết nối; input đọc được, output/cache ghi được.")

In [ ]:
# @title 3 · Tải / cập nhật source GitHub public
import re
import subprocess
from urllib.parse import urlparse
from pathlib import Path

url = urlparse(GITHUB_REPO)
if url.scheme != "https" or url.netloc != "github.com" or not re.fullmatch(r"/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?", url.path):
    raise ValueError("Dùng URL public dạng https://github.com/OWNER/REPO.git")
subprocess.run(["git", "check-ref-format", "--branch", BRANCH], check=True, capture_output=True)
if PIN_COMMIT and not re.fullmatch(r"[a-fA-F0-9]{40}", PIN_COMMIT):
    raise ValueError("PIN_COMMIT phải là SHA đầy đủ 40 ký tự, hoặc để rỗng.")
repo = Path(PROJECT_DIR)
env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
if not repo.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--", GITHUB_REPO, str(repo)], env=env, check=True)
origin = subprocess.check_output(["git", "-C", str(repo), "remote", "get-url", "origin"], text=True).strip()
if origin.removesuffix(".git") != GITHUB_REPO.removesuffix(".git"):
    raise ValueError("PROJECT_DIR chứa repository khác; dùng runtime mới.")
if subprocess.check_output(["git", "-C", str(repo), "status", "--porcelain"], text=True).strip():
    raise ValueError("Source local đã sửa. Lưu thay đổi trước khi cập nhật hoặc dùng runtime mới.")
subprocess.run(["git", "-C", str(repo), "fetch", "origin", BRANCH], env=env, check=True)
if PIN_COMMIT:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", PIN_COMMIT], env=env, check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "--detach", PIN_COMMIT], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", BRANCH], env=env, check=True)
subprocess.run(["git", "-C", str(repo), "log", "-1", "--format=%h %s"], check=True)

In [ ]:
# @title 4 · Cài đặt và kiểm tra môi trường
subprocess.run(["bash", "scripts/setup_colab.sh"], cwd=PROJECT_DIR, check=True)
PYTHON = str(Path(PROJECT_DIR) / ".venv-colab/bin/python")
subprocess.run([PYTHON, "-m", "manga_colab", "doctor", "--input", INPUT_DIR,
                "--output", OUTPUT_DIR, "--models", MODEL_CACHE_DIR], cwd=PROJECT_DIR, check=True)

In [ ]:
# @title 5 · Chạy upscale
if not AUTO_MODEL:
    raise ValueError("Phiên bản này tự chọn model cho từng ảnh; hãy bật AUTO_MODEL.")
args = [PYTHON, "-u", "-m", "manga_colab", "run", "--input", INPUT_DIR,
        "--output", OUTPUT_DIR, "--models", MODEL_CACHE_DIR, "--scale", str(UPSCALE_FACTOR),
        "--format", OUTPUT_FORMAT, "--threshold", str(GRAYSCALE_THRESHOLD),
        "--detector", DETECTOR, "--tile", str(TILE_SIZE), "--precision", PRECISION,
        "--log-level", LOG_LEVEL, "--auto-model", "--resume" if RESUME else "--no-resume"]
for enabled, flag in ((OVERWRITE, "--overwrite"), (RETRY_FAILED, "--retry-failed"), (DRY_RUN, "--dry-run")):
    if enabled:
        args.append(flag)
if STAGING:
    args.extend(["--stage", "/content/manga-stage"])
result = subprocess.run(args, cwd=PROJECT_DIR,
                        env={**os.environ, "PYTHONUNBUFFERED": "1", "MANGA_COLAB_LIVE": "1", "COLUMNS": "110"})
if result.returncode:
    print(f"CLI exit code {result.returncode}. Xem nguyên nhân ở log và cell tổng kết; ảnh đã hoàn tất được giữ lại.")

In [ ]:
# @title 6 · Tổng kết
import json
from IPython.display import display, Markdown
from html import escape

report_path = Path(OUTPUT_DIR) / ".manga-colab/report.json"
if DRY_RUN:
    print("Dry-run plan:", Path(OUTPUT_DIR) / ".manga-colab/plan.json")
elif report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    rows = "\n".join(f"| {escape(str(key))} | {escape(str(value))} |" for key, value in report["summary"].items())
    display(Markdown("| Thống kê | Giá trị |\n|---|---|\n" + rows))
    print("Output:", OUTPUT_DIR)
    print("JSON / CSV / logs:", report_path.parent)
else:
    print("Chưa có report. Xem lỗi ở cell cài đặt/chạy phía trên.")